# 03 · Gold model

Builds the business-ready star schema that the Power BI semantic model reads.

| Table | Grain |
|---|---|
| `dim_date` | one row per calendar day |
| `dim_customer` | one row per customer |
| `dim_product` | one row per item |
| `fact_sales` | one row per order line, with cost and margin |
| `fact_production` | one row per production run, with the parts of OEE |
| `fact_inventory` | one row per item per weekly snapshot, with days on hand |
| `fact_downtime` | one row per downtime event |
| `fact_sales_weather_daily` | one row per region per day: sales next to that day's weather |

Gold is small, so every table is rebuilt in full on each run. That keeps the logic simple and always consistent with silver.

In [ ]:
%run 00_utils

In [ ]:
products = spark.table("silver_products")
customers = spark.table("silver_customers")
headers = spark.table("silver_sales_order_headers")
lines = spark.table("silver_sales_order_lines")
inventory = spark.table("silver_inventory_snapshots")
runs = spark.table("silver_production_runs")
downtime = spark.table("silver_downtime_events")
weather = spark.table("silver_weather_daily") if table_exists("silver_weather_daily") else None


def date_key(col):
    return F.date_format(col, "yyyyMMdd").cast("int")

## Dimensions

In [ ]:
bounds = (
    headers.select(F.col("order_date").alias("d"))
    .union(runs.select("production_date"))
    .union(inventory.select("snapshot_date"))
    .agg(F.min("d").alias("lo"), F.max("d").alias("hi"))
    .first()
)
first_day = date(bounds.lo.year, 1, 1)
last_day = date(bounds.hi.year, 12, 31)

dim_date = (
    spark.sql(f"SELECT explode(sequence(DATE'{first_day}', DATE'{last_day}', INTERVAL 1 DAY)) AS date")
    .select(
        date_key(F.col("date")).alias("date_key"),
        "date",
        F.year("date").alias("year"),
        F.quarter("date").alias("quarter"),
        F.month("date").alias("month"),
        F.date_format("date", "MMM").alias("month_name"),
        F.date_trunc("week", "date").cast("date").alias("week_start"),
        F.date_format("date", "EEE").alias("day_name"),
        F.dayofweek("date").isin(1, 7).alias("is_weekend"),
    )
)

dim_customer = customers.select("customer_no", "customer_name", "channel", "region", "payment_terms_days")
dim_product = products.select("item_no", "item_name", "product_family", "case_price", "unit_cost", "cases_per_pallet")

## Facts

In [ ]:
# Sales: one row per order line. Cost uses the item's current standard cost.
fact_sales = (
    lines.alias("l")
    .join(headers.alias("h"), "order_no")
    .join(products.select("item_no", "unit_cost").alias("p"), "item_no")
    .select(
        "order_no", "line_no",
        date_key(F.col("h.order_date")).alias("order_date_key"),
        F.col("h.order_date").alias("order_date"),
        F.col("h.requested_ship_date").alias("requested_ship_date"),
        F.col("h.customer_no").alias("customer_no"),
        "item_no",
        F.col("h.status").alias("status"),
        F.col("l.cases").alias("cases"),
        (F.col("l.cases") * F.col("l.unit_price")).cast("decimal(14,2)").alias("gross_amount"),
        F.col("l.net_amount").alias("net_amount"),
        (F.col("l.cases") * F.col("p.unit_cost")).cast("decimal(14,2)").alias("cost_amount"),
    )
    .withColumn("discount_amount", (F.col("gross_amount") - F.col("net_amount")).cast("decimal(14,2)"))
    .withColumn("margin_amount", (F.col("net_amount") - F.col("cost_amount")).cast("decimal(14,2)"))
)

# Production: OEE is stored as additive parts so it rolls up correctly in Power BI
# (OEE for a month = sum of good cases / sum of ideal cases at planned time, not an average of daily percentages).
fact_production = (
    runs.withColumn("run_minutes", F.col("planned_minutes") - F.col("downtime_minutes"))
    .select(
        "run_id",
        date_key(F.col("production_date")).alias("production_date_key"),
        "production_date", "line", "item_no",
        "planned_minutes", "downtime_minutes", "run_minutes",
        (F.col("run_minutes") * F.col("ideal_cases_per_minute")).alias("ideal_cases_at_run_time"),
        (F.col("planned_minutes") * F.col("ideal_cases_per_minute")).alias("ideal_cases_at_planned_time"),
        "total_cases", "good_cases",
    )
    .withColumn("availability", F.col("run_minutes") / F.col("planned_minutes"))
    .withColumn("performance", F.col("total_cases") / F.col("ideal_cases_at_run_time"))
    .withColumn("quality", F.col("good_cases") / F.col("total_cases"))
    .withColumn("oee", F.col("good_cases") / F.col("ideal_cases_at_planned_time"))
)

# Inventory: days on hand = available cases / average cases ordered per day over the previous 28 days.
daily_demand = fact_sales.groupBy("item_no", "order_date").agg(F.sum("cases").alias("cases"))
demand_28d = (
    inventory.select("snapshot_date", "item_no").alias("i")
    .join(
        daily_demand.alias("d"),
        (F.col("i.item_no") == F.col("d.item_no"))
        & (F.col("d.order_date") >= F.date_sub("i.snapshot_date", 28))
        & (F.col("d.order_date") < F.col("i.snapshot_date")),
        "left",
    )
    .groupBy("i.snapshot_date", "i.item_no")
    .agg((F.coalesce(F.sum("d.cases"), F.lit(0)) / 28).alias("avg_daily_cases_28d"))
)
fact_inventory = (
    inventory.join(demand_28d, ["snapshot_date", "item_no"], "left")
    .select(
        date_key(F.col("snapshot_date")).alias("snapshot_date_key"),
        "snapshot_date", "item_no", "warehouse",
        "cases_on_hand", "cases_on_hold_qa",
        (F.col("cases_on_hand") - F.col("cases_on_hold_qa")).alias("cases_available"),
        F.round("avg_daily_cases_28d", 2).alias("avg_daily_cases_28d"),
    )
    .withColumn(
        "days_on_hand",
        F.when(F.col("avg_daily_cases_28d") > 0, F.round(F.col("cases_available") / F.col("avg_daily_cases_28d"), 1)),
    )
)

fact_downtime = downtime.select(
    "event_id", "run_id",
    date_key(F.col("production_date")).alias("production_date_key"),
    "production_date", "line", "reason", "category", "minutes",
)

# Sales and weather side by side, per region and day (weekends have weather but no orders).
sales_by_region_day = (
    fact_sales.join(customers.select("customer_no", "region"), "customer_no")
    .groupBy("region", F.col("order_date").alias("date"))
    .agg(F.sum("net_amount").alias("net_amount"), F.sum("cases").alias("cases"))
)

## Write

In [ ]:
gold = {
    "dim_date": dim_date,
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "fact_sales": fact_sales,
    "fact_production": fact_production,
    "fact_inventory": fact_inventory,
    "fact_downtime": fact_downtime,
}

if weather is not None:
    gold["fact_sales_weather_daily"] = (
        weather.select("region", "city", F.col("weather_date").alias("date"),
                       F.col("temperature_2m_max").alias("temp_max_f"),
                       F.col("temperature_2m_min").alias("temp_min_f"),
                       F.col("precipitation_sum").alias("precipitation_in"))
        .join(sales_by_region_day, ["region", "date"], "left")
        .select(date_key(F.col("date")).alias("date_key"), "date", "region", "city",
                F.coalesce("net_amount", F.lit(0).cast("decimal(14,2)")).alias("net_amount"),
                F.coalesce("cases", F.lit(0)).alias("cases"),
                "temp_max_f", "temp_min_f", "precipitation_in")
    )

for name, df in gold.items():
    overwrite(df, name)
    log(f"  {name:<26} {spark.table(name).count():>9,} rows")